In [ ]:
import os
os.chdir('/kaggle/working/')

#if not os.path.exists('/kaggle/tmp'):
#    os.mkdir('/kaggle/tmp')
#os.chdir('/kaggle/tmp/')

print(os.getcwd())

import torch
print(torch.cuda.is_available())  # Should return True if GPU is available
print(torch.cuda.device_count())  # Should return 1 on Kaggle

In [ ]:
import subprocess
import os

def run(commands):
    for command in commands:
        with subprocess.Popen(command, shell = True, stdout = subprocess.PIPE, stderr = subprocess.STDOUT, bufsize = 1) as sp:
            for line in sp.stdout:
                line = line.decode("utf-8", errors = "replace")
                if "undefined reference" in line:
                    raise RuntimeError("Failed Processing.")
                print(line, flush = True, end = "")
        pass
    pass
pass
commands = [
        "curl -fsSL https://ollama.com/install.sh | sh",
]
run(commands)


In [ ]:
import os
os.system("/usr/local/bin/ollama serve &")
os.system("echo 'ollama test'")
commands = [
        #"telnet 0.0.0.0 11434"
        #"ollama pull llama3",
        #"ollama pull phi3",
        #"ollama pull mistral",
        #"ollama run llama3 \"create 10 sentences that ends with apple\""
        "ollama run gemma2:27b \"create 10 sentences that ends with apple\""
        #"curl http://127.0.0.1:11434/api/chat -d '{\"model\": \"llama3\", \"stream\": false, \"messages\": [{ \"role\": \"user\", \"content\": \"create 10 sentences that ends with apple\" }]}'"
]
run(commands)

In [ ]:
import requests
import json
import pandas as pd
import csv
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_expansion(row):
    url = "http://127.0.0.1:11434/api/chat"
    headers = {
        "Content-Type": "application/json"
    }
    
    prompt = f"""
    YOU ARE A MINIMAL SPELL CORRECTOR FOR PRODUCT DESCRIPTIONS. YOUR TASK IS TO MAKE ONLY ABSOLUTELY NECESSARY CORRECTIONS:
    1. DO NOT CHANGE ANY WORDS OR NUMBERS.
    2. DO NOT ADD OR REMOVE ANY INFORMATION.
    3. ONLY CORRECT OBVIOUS SPELLING ERRORS.
    4. YOU MAY ADD HYPHENS ONLY IF ABSOLUTELY NECESSARY FOR CLARITY.
    5. MAINTAIN ALL ABBREVIATIONS AND PRODUCT NAMES AS THEY ARE.
    
    Examples:
    1. Original: "1 cp rmx screenwash"
       Refined:  "1 cp rmx screenwash"
       (No changes needed)

    2. Original: "ad blue 5 litre 000602"
       Refined:  "adblue 5 litre 000602"
       (Only joined "adblue" as it's a known product name)

    3. Original: "airwick car berrjes"
       Refined:  "airwick car berries"
       (Only corrected spelling of "berries")

    4. Original: "all round wipfs 20 s"
       Refined:  "all round wipes 20 s"
       (Only added hyphen for clarity)

    Now, refine the following description:
    Context:
    Supergroup: {row['supergroup']}
    Group: {row['group']}
    Module: {row['module']}
    Brand: {row['brand']}
    Retailer: {row['retailer']}
    
    Original description: "{row['description']}"
    
    PROVIDE ONLY THE SINGLE-LINE REFINED DESCRIPTION. MAKE MINIMAL CHANGES.
    """
    
    data = {
        "model": "gemma2:27b",
        "messages": [{
            "role": "user",
            "content": prompt
        }],
        "stream": False
    }
    
    try:
        response = requests.post(url, headers=headers, json=data, timeout=10)
        response.raise_for_status()
        return response.json()["message"]["content"].strip()
    except requests.exceptions.RequestException as e:
        print(f"Error in API call: {e}")
        return None

def process_descriptions(input_file, output_directory, start_row, end_row):
    try:
        df = pd.read_csv(input_file)
        
        required_columns = ['description', 'supergroup', 'group', 'module', 'brand', 'retailer']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"One or more required columns not found in the input CSV file. Required: {required_columns}")
        
        output_file = os.path.join(output_directory, 'output_refined_TEST_descriptions.csv')
        df_subset = df.iloc[start_row:min(end_row, len(df))].copy()
        
        # Create the output CSV file and write the header
        with open(output_file, 'w', newline='') as csvfile:
            fieldnames = list(df_subset.columns) + ['refined_description']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
        
        def process_and_save_row(row):
            refined = get_expansion(row)
            if refined:
                row_dict = row.to_dict()
                row_dict['refined_description'] = refined
                with open(output_file, 'a', newline='') as csvfile:
                    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                    writer.writerow(row_dict)
                print(f"Processed and saved description for index {row.name}")
            return refined
        
        with ThreadPoolExecutor(max_workers=5) as executor:
            future_to_index = {
                executor.submit(process_and_save_row, row): idx 
                for idx, row in df_subset.iterrows()
            }
            
            for future in as_completed(future_to_index):
                idx = future_to_index[future]
                try:
                    future.result()
                except Exception as e:
                    print(f"Error processing description at index {idx}: {e}")
        
        print(f"Refinement complete! Processed rows {start_row} to {end_row}. Saved to '{output_file}'")
    except Exception as e:
        print(f"Error in process_descriptions: {e}")

# Usage
input_file = "/kaggle/input/indo-new-test-data/final_test_data.csv"
output_directory = "/kaggle/working"
start_row = 90001
end_row = 120000

process_descriptions(input_file, output_directory, start_row, end_row)

In [ ]:
import pandas
op = pandas.read_csv('/kaggle/working/output_refined_descriptions_2.csv')
op.shape

In [ ]:
%cd /kaggle/working

In [ ]:
!ls /kaggle/working

In [ ]:
from IPython.display import FileLink 
FileLink('output_refined_descriptions.csv')